# MFCC-BiLSTM

Mel-Frequency Cepstral Coefficients fed into a Bidirectional LSTM classifier.
The feature tensor shape is `(FIXED_LEN, N_MFCC)` — time steps are the sequence dimension.

## 1. Imports & Config

In [1]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


In [2]:
ROOT_PATH = "/kaggle/input/datasets/artharking/severity-dau-kdah/Severity_DAU_KDAH"  # change this

SR = 16000
N_MFCC = 20          # number of MFCC coefficients
FRAME_LEN = 0.025     # 25 ms
HOP_LEN   = 0.010     # 10 ms

FIXED_LEN  = 398      # time frames

# BiLSTM hyper-params
HIDDEN_SIZE = 256
NUM_LAYERS  = 2
DROPOUT     = 0.3

BATCH_SIZE = 32
EPOCHS     = 20
LR         = 1e-3


In [3]:
label_map = {
    "N": 0,
    "L": 1,
    "M": 2,
    "H": 3
}
NUM_CLASSES = len(label_map)

In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


## 2. Feature Extraction — MFCC

In [5]:
def extract_mfcc(file_path):
    y, sr = librosa.load(file_path, sr=SR)
    mfcc  = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=N_MFCC,
        n_fft=int(FRAME_LEN * sr),
        hop_length=int(HOP_LEN * sr)
    )
    if mfcc.shape[1] < FIXED_LEN:
        mfcc = np.pad(mfcc, ((0,0),(0,FIXED_LEN-mfcc.shape[1])), mode='constant')
    else:
        mfcc = mfcc[:, :FIXED_LEN]
    return mfcc   # (N_MFCC, FIXED_LEN)


## 3. Precompute & Save Features

In [6]:
SAVE_PATH = "/kaggle/working/mfcc_features"
os.makedirs(SAVE_PATH, exist_ok=True)

def save_mfcc_dataset(root_dir, split):
    split_path = os.path.join(root_dir, split)
    for severity in os.listdir(split_path):
        sev_path = os.path.join(split_path, severity)
        if not os.path.isdir(sev_path): continue
        save_sev = os.path.join(SAVE_PATH, split, severity)
        os.makedirs(save_sev, exist_ok=True)
        for file in tqdm(os.listdir(sev_path), desc=f'{split}-{severity}'):
            if not file.endswith('.wav'): continue
            np.save(os.path.join(save_sev, file.replace('.wav','.npy')),
                    extract_mfcc(os.path.join(sev_path, file)))

save_mfcc_dataset(ROOT_PATH, 'train')
save_mfcc_dataset(ROOT_PATH, 'test')


test-L: 100%|██████████| 79/79 [00:01<00:00, 57.93it/s]


## 4. Dataset & DataLoaders

In [7]:
import torch
from torch.utils.data import Dataset

class MfccDataset(Dataset):
    def __init__(self, root_dir):
        self.files  = []
        self.labels = []
        for severity in os.listdir(root_dir):
            sev_path = os.path.join(root_dir, severity)
            if not os.path.isdir(sev_path): continue
            for file in os.listdir(sev_path):
                if file.endswith('.npy'):
                    self.files.append(os.path.join(sev_path, file))
                    self.labels.append(label_map[severity])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        mfcc = np.load(self.files[idx])          # (N_FEAT, FIXED_LEN)
        # BiLSTM expects (seq_len, input_size) → transpose to (FIXED_LEN, N_FEAT)
        mfcc = torch.tensor(mfcc.T, dtype=torch.float32)  # (FIXED_LEN, N_FEAT)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mfcc, label


In [8]:
train_dataset = MfccDataset("/kaggle/working/mfcc_features/train")
test_dataset  = MfccDataset("/kaggle/working/mfcc_features/test")

val_size   = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_ds, val_ds = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 5. BiLSTM Model

In [9]:
import torch
import torch.nn as nn

drop_amount = 0.255

class BiLSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=drop_amount if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(drop_amount)

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):

        # Initial hidden and cell states
        h0 = torch.zeros(
            self.num_layers * 2,
            x.size(0),
            self.hidden_size,
            device=x.device
        )

        c0 = torch.zeros(
            self.num_layers * 2,
            x.size(0),
            self.hidden_size,
            device=x.device
        )

        # LSTM output
        out, _ = self.lstm(x, (h0, c0))

        out = self.dropout(out)

        # Forward last timestep + backward first timestep
        last_hidden_state = torch.cat(
            (
                out[:, -1, :self.hidden_size],
                out[:, 0, self.hidden_size:]
            ),
            dim=1
        )

        output = self.fc(last_hidden_state)

        return output

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMClassifier(
    input_size=N_MFCC,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES
).to(device)


## 6. Training

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

import shutil

best_val_acc = 0
save_path = "/tmp/best_model.pth"

for epoch in range(EPOCHS):

    # ================= TRAIN =================
    model.train()
    train_loss = 0

    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [train]'):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    # ================= VALIDATION =================
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()

            total += y.size(0)

    val_acc = correct / total

    print(f'Epoch {epoch+1}/{EPOCHS}: Loss={train_loss:.4f}  Val Acc={val_acc:.4f}')

    # ================= SAVE BEST =================
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            save_path,
            _use_new_zipfile_serialization=False
        )

        print(f'✅ Best model saved (Val Acc={val_acc:.4f})')

shutil.copy(save_path, './best_model.pth')

print('✅ Model copied to working directory!')

Model params: 2,148,356


Epoch 1/20 [train]: 100%|██████████| 289/289 [00:23<00:00, 12.09it/s]


Epoch 1/20: Loss=100.8529  Val Acc=0.8771
✅ Best model saved (Val Acc=0.8771)


Epoch 2/20 [train]: 100%|██████████| 289/289 [00:23<00:00, 12.36it/s]


Epoch 2/20: Loss=79.9592  Val Acc=0.9180
✅ Best model saved (Val Acc=0.9180)


...

Epoch 18/20: Loss=16.6104  Val Acc=0.9659


Epoch 19/20 [train]: 100%|██████████| 289/289 [00:23<00:00, 12.34it/s]


Epoch 19/20: Loss=19.0287  Val Acc=0.9737


Epoch 20/20 [train]: 100%|██████████| 289/289 [00:23<00:00, 12.35it/s]


Epoch 20/20: Loss=16.5511  Val Acc=0.9824
✅ Best model saved (Val Acc=0.9824)
✅ Model copied to working directory!


## 7. Evaluation

In [12]:
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

correct = total = 0
all_preds  = []
all_labels = []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc='Test'):
        x, y  = x.to(device), y.to(device)
        preds = torch.argmax(model(x), dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

print(f'✅ Test Accuracy: {correct/total:.4f}')


Test: 100%|██████████| 84/84 [00:02<00:00, 31.51it/s]

✅ Test Accuracy: 0.9247


In [13]:
from sklearn.metrics import classification_report
print('\nClassification Report:\n')
print(classification_report(all_labels, all_preds, target_names=list(label_map.keys())))



Classification Report:

              precision    recall  f1-score   support

           N       0.98      1.00      0.99      2037
           L       0.30      0.43      0.35        79
           M       0.82      0.81      0.82       506
           H       0.00      0.00      0.00        59

    accuracy                           0.92      2681
   macro avg       0.53      0.56      0.54      2681
weighted avg       0.91      0.92      0.92      2681



In [14]:
# import pandas as pd

# inv_map = {v: k for k, v in label_map.items()}

# filenames = [os.path.basename(test_dataset.files[i]) for i in range(len(test_dataset))]

# scores = []
# all_preds = []
# all_labels = []

# model.eval()
# with torch.no_grad():
#     for x, y in tqdm(DataLoader(test_dataset, batch_size=32, shuffle=False)):
#         x, y = x.to(device), y.to(device)
#         outputs = model(x)
#         probs = torch.softmax(outputs, dim=1)
#         best_scores = probs.max(dim=1).values
#         preds = torch.argmax(outputs, dim=1)

#         scores.extend(best_scores.cpu().numpy())
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(y.cpu().numpy())

# df = pd.DataFrame({
#     "filename":      filenames,
#     "score":         scores,
#     "predict class": [inv_map[p] for p in all_preds],
#     "actual class":  [inv_map[l] for l in all_labels],
# })

# df.to_excel("/kaggle/working/results.xlsx", index=False)
# print("✅ Saved results.xlsx")

import pandas as pd

inv_map = {v: k for k, v in label_map.items()}

# Class order: Normal → High → Mid → Low → Very_Low
class_order = ["N", "L", "M", "H"]

filenames  = [os.path.basename(test_dataset.files[i]) for i in range(len(test_dataset))]

all_preds      = []
all_labels     = []
all_raw_scores = []   # raw logits per class
all_softmax    = []   # softmax probabilities per class

model.eval()
with torch.no_grad():
    for x, y in tqdm(DataLoader(test_dataset, batch_size=32, shuffle=False)):
        x, y = x.to(device), y.to(device)
        outputs = model(x)                          # raw logits: (batch, NUM_CLASSES)
        probs   = torch.softmax(outputs, dim=1)     # softmax probabilities
        preds   = torch.argmax(outputs, dim=1)

        all_raw_scores.extend(outputs.cpu().numpy())
        all_softmax.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Build DataFrame
df = pd.DataFrame({"filename": filenames})

# Raw logit score for each class
for cls_name in class_order:
    df[f"score_{cls_name}"] = [row[label_map[cls_name]] for row in all_raw_scores]

# Softmax probability for each class
for cls_name in class_order:
    df[f"prob_{cls_name}"] = [row[label_map[cls_name]] for row in all_softmax]

# Final prediction & actual label
df["predict class"] = [inv_map[p] for p in all_preds]
df["actual class"]  = [inv_map[l] for l in all_labels]

# Save both CSV and Excel
df.to_csv("/kaggle/working/results.csv",   index=False)
df.to_excel("/kaggle/working/results.xlsx", index=False)
print("✅ Saved results.csv and results.xlsx")
print(df.head())


100%|██████████| 84/84 [00:02<00:00, 31.79it/s]


✅ Saved results.csv and results.xlsx
                 filename   score_N   score_L   score_M   score_H    prob_N  \
0  M03_English_L_0064.npy -3.707716  3.452923  0.690032 -3.162065  0.000729   
1  M03_English_L_0026.npy -2.090956  2.205609  1.049537 -4.529327  0.010241   
2  M09_English_L_0001.npy -5.598135  0.562054  2.882964  1.293350  0.000159   
3  M03_English_L_0017.npy -2.211282  2.399913  1.172770 -5.058635  0.007625   
4  M03_English_L_0008.npy -2.003186  0.737346  2.369935 -4.885605  0.010434   

     prob_L    prob_M    prob_H predict class actual class  
0  0.938768  0.059245  0.001258             L            L  
1  0.752149  0.236716  0.000894             L            L  
2  0.075387  0.767816  0.156638             M            L  
3  0.767080  0.224853  0.000442             L            L  
4  0.161675  0.827307  0.000584             M            L  


## 8. Single-File Prediction

In [15]:
# def predict(file_path, model_path='best_model.pth'):
#     m = BiLSTMModel(N_MFCC, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT)
#     m.load_state_dict(torch.load(model_path, map_location='cpu'))
#     m.eval()
#     feat = torch.tensor(extract_mfcc(file_path).T, dtype=torch.float32).unsqueeze(0)
#     with torch.no_grad():
#         pred = torch.argmax(m(feat), dim=1).item()
#     return {v:k for k,v in label_map.items()}[pred]
